In [1]:
import pandas as pd
import numpy as np
import time

# Read input data
data = pd.read_excel('/Users/workk/Documents/[Think!]/[FMK]/Research Things/Economic Dispatch/Datasets/Jamali/Same System/input42u.xlsx')
a = data['a'].values
b = data['b'].values
c = data['c'].values
Minimum_Capacity = data['pmin'].values
Maximum_Capacity = data['pmax'].values
ramp_up = data['rampup'].values
ramp_down = data['rampdown'].values
Unit = data['Unit'].values

# Read power demand data
power_demand_data = pd.read_excel('/Users/workk/Documents/[Think!]/[FMK]/Research Things/Economic Dispatch/Datasets/Jamali//Same System/power_demand42u.xlsx')
power_demand = power_demand_data['load'].values

# Initialize an empty list to store results for each demand
all_outputs = []

# Define function to calculate power produced using matrix operations
def calculate_power(demand, a, b, c, pmin, pmax, rampup, rampdown):
    Lambda = max(b)
    P = np.zeros(len(pmin))
    iterations = 0

    while abs(np.subtract(demand, np.sum(P))) > 0.00001:
        iterations += 1
        multiplier = np.divide(np.subtract(Lambda, b), 2)
        P_new = np.minimum(np.maximum(np.divide(multiplier, c), pmin), pmax)
        Power_Demand = np.subtract(demand, np.sum(P_new))

        if np.sum(P) > 0:
            diff = np.subtract(P_new, P)
            ramp_up_mask = diff > 0
            ramp_down_mask = diff < 0
            ramp_up_values = np.minimum(diff[ramp_up_mask], rampup[ramp_up_mask])
            ramp_down_values = np.maximum(diff[ramp_down_mask], -rampdown[ramp_down_mask])
            update = lambda x, y: np.add(x, y)
            P_new[ramp_up_mask] = update(P[ramp_up_mask], ramp_up_values)
            P_new[ramp_down_mask] = update(P[ramp_down_mask], ramp_down_values)

        P = P_new
        Lambda = np.add(Lambda, (Power_Demand / (np.sum(np.divide(1, np.multiply(2, c))))))

    return P, iterations

# Define function to calculate fuel cost using matrix operations
def calculate_fuel_cost(P, a, b, c):
    squaredP = np.multiply(P, P)
    Fuel_Cost = np.divide(np.add(a, np.add(np.multiply(b, P), np.multiply(c, squaredP))),1000)
    total_Fuel_Cost = np.sum(Fuel_Cost)
    return Fuel_Cost, total_Fuel_Cost

# Loop through power demand data and calculate results
all_outputs = []
schedulling = []

for load_counter, demand in enumerate(power_demand, start=1):
    start_time = time.time()
    if load_counter == 1:
        previous_output = None
    else:
        previous_output = all_outputs[-1]['Output']['Power Produced (MW)'].values
    P, iterations = calculate_power(demand, a, b, c, Minimum_Capacity, Maximum_Capacity, ramp_up, ramp_down)
    Fuel_Cost, total_Fuel_Cost = calculate_fuel_cost(P, a, b, c)
    end_time = time.time()

    # Output Results
    output = pd.DataFrame({'Unit': Unit, 'Power Produced (MW)': P})
    print("Load Counter: {}".format(load_counter))
    print(f"Output for Power Demand {demand:.2f} MW")
    print(output)
    print(f"Total Power Produced: {np.sum(P):.2f} MW")
    print(f'Total Fuel Cost: Rp {total_Fuel_Cost}')
    print(f'Computation Time: {end_time - start_time} seconds')
    print('------------------------------------------------\n')

    all_outputs.append({'Demand': demand, 
                        'Output': output, 
                        'Total Power Produced': np.sum(P),
                        'Total Fuel Cost': total_Fuel_Cost,  # No need to sum again, it's already a single value
                        'Computation Time': end_time - start_time})

    # Scheduling
    output_info = {'Demand': power_demand[load_counter - 1]}
    for unit_index, unit in enumerate(Unit):
        output_info[f'Unit {unit} Power Produced'] = P[unit_index]
        output_info[f'Unit {unit} Cost'] = total_Fuel_Cost  # Assign total cost directly
    schedulling.append(output_info)

# Calculate ramp rates and save to an additional sheet
ramp_rates = []
generation_limits = []
for i in range(1, len(power_demand)):  # Start from 1 because there's no previous output for the first demand
    previous_output = all_outputs[i-1]['Output']['Power Produced (MW)'].values
    current_output = all_outputs[i]['Output']['Power Produced (MW)'].values
    ramp_rate = current_output - previous_output

    # Check for ramp rate violations
    violations = np.logical_or(ramp_rate > ramp_up, ramp_rate < -ramp_down)
    ramp_rate_info = {'Demand': power_demand[i]}
    generation_limit_info = {'Demand': power_demand[i]}
    for unit_index, unit in enumerate(Unit):
        ramp_rate_info[f'Unit {unit}'] = ramp_rate[unit_index]
        ramp_rate_info[f'Unit {unit} Violation'] = violations[unit_index]
               
        # Check for generation limit violations
        gen_violation = (current_output[unit_index] < Minimum_Capacity[unit_index]) or (current_output[unit_index] > Maximum_Capacity[unit_index])
        generation_limit_info[f'Unit {unit}'] = current_output[unit_index]
        generation_limit_info[f'Unit {unit} Violation'] = gen_violation
    ramp_rates.append(ramp_rate_info)
    generation_limits.append(generation_limit_info)
# Save results to Excel
output_file_path = '/Users/workk/Documents/[Think!]/[FMK]/Research Things/Economic Dispatch/Output/VLIM.xlsx'
with pd.ExcelWriter(output_file_path) as writer:
    pd.DataFrame(schedulling).to_excel(writer, sheet_name='Schedulling', index=False)
    pd.DataFrame(all_outputs).to_excel(writer, sheet_name='Summary', index=False)
    pd.DataFrame(ramp_rates).to_excel(writer, sheet_name='Ramp Rates', index=False)
    pd.DataFrame(generation_limits).to_excel(writer, sheet_name='Generation Limits', index=False)
print("All results saved to 'VLIM.xlsx'")

Load Counter: 1
Output for Power Demand 8761.42 MW
    Unit  Power Produced (MW)
0      1           250.000000
1      2           250.000000
2      3           250.000000
3      4           250.000000
4      5           408.000000
5      6           408.000000
6      7           408.000000
7      8           174.600000
8      9           174.600000
9     10           174.600000
10    11           174.600000
11    12            80.000000
12    13            80.000000
13    14           157.011995
14    15           157.011995
15    16           220.000000
16    17           220.000000
17    18           220.000000
18    19            16.000000
19    20            16.000000
20    21           118.500000
21    22           118.500000
22    23           118.500000
23    24           118.500000
24    25           118.500000
25    26           118.500000
26    27           118.500000
27    28           118.500000
28    29           400.000000
29    30           400.000000
30    31           